# Violence pipeline: COLAB 2 of 3 (shards 1,4)
Settings are already filled in for Colab 2: do not edit, just Run all.
Runtime > Change runtime type > **L4 GPU**, then Runtime > Run all.
After a disconnect just Run all again: finished cells are skipped (DONE) and the cell in progress
resumes from its `last.pt` on Drive.

In [ ]:
# ---- user settings ----
SHARDS = "1,4"          # this machine's shards (laptop runs 0/7; colab-1: "2,5", colab-2: "1,4", colab-3: "3,6";
                        # balanced from the timing pilot: ~24.5 laptop-GPU hours per Colab)
N_SHARDS = 7
# The owner's shared Drive folder (holds repo.zip, cache.zip, splits/, bench/ and runs/).
# Friends: in Drive, "Add shortcut to Drive" on the owner's shared folder and place it in
# My Drive under exactly this name.
DRIVE_DIR = "/content/drive/MyDrive/violence_pipeline"
MACHINE = "colab-2"     # label for the log file on Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir(DRIVE_DIR), f"{DRIVE_DIR} not found: add the shared folder shortcut first"
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > L4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
import os, shutil, subprocess
# Code is small: always refresh it from Drive. The frame cache is ~4.5 GB: unzip it once per runtime.
subprocess.run(["unzip", "-q", "-o", f"{DRIVE_DIR}/repo.zip", "-d", "/content/repo"], check=True)
os.makedirs("/content/work", exist_ok=True)
if not os.path.exists("/content/work/.cache_ok"):
    subprocess.run(["unzip", "-q", "-o", f"{DRIVE_DIR}/cache.zip", "-d", "/content/work"], check=True)
    open("/content/work/.cache_ok", "w").close()
shutil.copytree(f"{DRIVE_DIR}/splits", "/content/work/splits", dirs_exist_ok=True)  # locked splits + LOCK.json
if os.path.isdir(f"{DRIVE_DIR}/bench"):
    shutil.copytree(f"{DRIVE_DIR}/bench", "/content/work/bench", dirs_exist_ok=True)  # timing for --dry-run
print(open("/content/repo/GIT_COMMIT").read())

In [ ]:
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# Batch stays at the per-model defaults (same micro-batch as the laptop, effective batch 32), so a
# cell gives the same BatchNorm behaviour whichever machine trains it.
SETS = f'--set paths.work=/content/work --set "paths.runs={DRIVE_DIR}/runs"'
!python -u -m pipeline grid --shard {SHARDS}/{N_SHARDS} {SETS} 2>&1 | tee -a "{DRIVE_DIR}/logs/{MACHINE}.log"

In [ ]:
!python -u -m pipeline evaluate --all {SETS} 2>&1 | tee -a "{DRIVE_DIR}/logs/{MACHINE}.log"